# LMSYS Chatbot Arena — Baseline Notebook
End-to-end pipeline: load → preprocess → train → submit

This was a TD-IDF attempt. Public log loss of 1.10832.

## 1. Setup

In [1]:
# ── PATH SWITCHER ─────────────────────────────────────────────────────────────
# Comment out the environment you are NOT using

# ENV = "local"
ENV = "kaggle"

if ENV == "local":
    DATA_DIR   = "Data/"
    OUTPUT_DIR = "."

elif ENV == "kaggle":
    DATA_DIR   = "/kaggle/input/competitions/lmsys-chatbot-arena"
    OUTPUT_DIR = "/kaggle/working"

import os
TRAIN_PATH      = os.path.join(DATA_DIR, "train.csv")
TEST_PATH       = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")

print(f"ENV          : {ENV}")
print(f"TRAIN_PATH   : {TRAIN_PATH}")
print(f"SUBMISSION   : {SUBMISSION_PATH}")

ENV          : kaggle
TRAIN_PATH   : /kaggle/input/competitions/lmsys-chatbot-arena/train.csv
SUBMISSION   : /kaggle/working/submission.csv


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from scipy.sparse import hstack

RANDOM_SEED = 42
TARGET_COLS = ["winner_model_a", "winner_model_b", "winner_tie"]

## 2. Load and Inspect Data

In [3]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(f"train : {train.shape}")
print(f"test  : {test.shape}")
print()
print("train columns:", train.columns.tolist())
print("test  columns:", test.columns.tolist())
train.head(2)

train : (57477, 9)
test  : (3, 4)

train columns: ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']
test  columns: ['id', 'prompt', 'response_a', 'response_b']


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0


## 3. Preprocessing

In [4]:
# Create integer label from one-hot target columns
# 0 = winner_model_a | 1 = winner_model_b | 2 = winner_tie
train["label"] = np.argmax(
    train[["winner_model_a", "winner_model_b", "winner_tie"]].to_numpy(),
    axis=1
)

print(train[["winner_model_a", "winner_model_b", "winner_tie", "label"]].head())
print()
print(train["label"].value_counts())

   winner_model_a  winner_model_b  winner_tie  label
0               1               0           0      0
1               0               1           0      1
2               0               0           1      2
3               1               0           0      0
4               0               1           0      1

label
0    20064
1    19652
2    17761
Name: count, dtype: int64


In [5]:
def format_input(row):
    return (
        f"[PROMPT]\n{row['prompt']}\n\n"
        f"[RESPONSE A]\n{row['response_a']}\n\n"
        f"[RESPONSE B]\n{row['response_b']}"
    )

train["text"] = train.apply(format_input, axis=1)
test["text"]  = test.apply(format_input, axis=1)

print("Sample formatted input:")
print(train["text"].iloc[0][:400])

Sample formatted input:
[PROMPT]
["Is it morally right to try to have a certain percentage of females on managerial positions?","OK, does pineapple belong on a pizza? Relax and give me fun answer."]

[RESPONSE A]
["The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and disc


## 4. Model Training and Validation

In [6]:
train_df, val_df = train_test_split(
    train,
    test_size=0.1,
    random_state=RANDOM_SEED,
    stratify=train["label"]
)

print(f"train_df : {len(train_df):,}")
print(f"val_df   : {len(val_df):,}")

train_df : 51,729
val_df   : 5,748


In [7]:
# TF-IDF vectoriser — fit on train only
tfidf = TfidfVectorizer(max_features=50_000, sublinear_tf=True)
X_train = tfidf.fit_transform(train_df["text"])
X_val   = tfidf.transform(val_df["text"])
X_test  = tfidf.transform(test["text"])

y_train = train_df["label"].values
y_val   = val_df["label"].values

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

X_train : (51729, 50000)
X_val   : (5748, 50000)
X_test  : (3, 50000)


In [8]:
model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="lbfgs",
    random_state=RANDOM_SEED,
)
model.fit(X_train, y_train)
print("Training complete.")

Training complete.


In [9]:
# Validation log-loss (competition metric is log-loss)
val_probs = model.predict_proba(X_val)
val_logloss = log_loss(y_val, val_probs)
print(f"Validation log-loss: {val_logloss:.4f}")

Validation log-loss: 1.1154


## 5. Inference and Submission

In [10]:
test_probs = model.predict_proba(X_test)

submission = pd.DataFrame(test_probs, columns=TARGET_COLS)
submission.insert(0, "id", test["id"].values)

# Sanity checks
assert submission.shape[0] == len(test), "Row count mismatch"
assert not submission.isnull().any().any(), "Nulls in submission"
assert list(submission.columns) == list(sample_sub.columns), "Column mismatch"

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved to: {SUBMISSION_PATH}")
print(f"Shape   : {submission.shape}")
submission.head()

Saved to: /kaggle/working/submission.csv
Shape   : (3, 4)


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.204450,0.359090,0.436460
1,211333,0.447833,0.237402,0.314765
2,1233961,0.232892,0.605785,0.161323
